<a href="https://colab.research.google.com/github/BuluGuluuu/Nhom-8---NLP/blob/main/Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Bước 1: Đọc và Khám phá Dữ liệu JSON
Chúng ta sẽ tải dữ liệu Intent và Entity từ các tệp cục bộ.

In [1]:
import json
import pandas as pd

# Đọc tệp Intent
with open('/content/intent_recommend_1200.json', 'r', encoding='utf-8') as f:
    intent_data = json.load(f)

# Đọc tệp Entity
with open('/content/entity_dataset_400.json', 'r', encoding='utf-8') as f:
    entity_data = json.load(f)

print(f"Số lượng mẫu Intent: {len(intent_data)}")
print(f"Số lượng mẫu Entity: {len(entity_data)}")

Số lượng mẫu Intent: 1200
Số lượng mẫu Entity: 400


#### Hiển thị mẫu dữ liệu Intent

In [2]:
df_intent = pd.DataFrame(intent_data)
display(df_intent.head())
print("Các loại Intent:", df_intent['intent'].unique())

,text,intent
0,"cho mình hỏi, ăn gì cho đỡ chán không",recommend_dish
1,"bạn ơi, Tôi nên chọn món ăn nào nhỉ",recommend_dish
2,ăn gì nhanh gọn vậy,recommend_dish
3,gợi ý món dễ làm đi nha,recommend_dish
4,"cho mình hỏi, Tôi cần một gợi ý về món ăn với",recommend_dish


Các loại Intent: ['recommend_dish']


#### Hiển thị mẫu dữ liệu Entity

In [3]:
df_entity = pd.DataFrame(entity_data)
display(df_entity.head())

,text,entities
0,đậu hũ có hợp để làm sushi không,"[{'value': 'sushi', 'type': 'dish'}, {'value':..."
1,có dưa leo làm mì ý được không,"[{'value': 'mì ý', 'type': 'dish'}, {'value': ..."
2,có cà rốt thì làm mì ý sao,"[{'value': 'mì ý', 'type': 'dish'}, {'value': ..."
3,làm gà sốt cần rau xà lách không,"[{'value': 'gà sốt', 'type': 'dish'}, {'value'..."
4,salad gà nấu với rau xà lách có ngon không,"[{'value': 'salad gà', 'type': 'dish'}, {'valu..."


### Bước 1.1: Tiền xử lý văn bản (Text Preprocessing)
Chúng ta định nghĩa hàm để làm sạch dữ liệu trước khi đưa vào mô hình.

In [4]:
import re
import string

def preprocess_text(text):
    # Chuyển về chữ thường
    text = text.lower()
    # Loại bỏ dấu câu
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Áp dụng tiền xử lý cho toàn bộ tập dữ liệu Intent
df_intent['text_cleaned'] = df_intent['text'].apply(preprocess_text)

display(df_intent[['text', 'text_cleaned']].head())

,text,text_cleaned
0,"cho mình hỏi, ăn gì cho đỡ chán không",cho mình hỏi ăn gì cho đỡ chán không
1,"bạn ơi, Tôi nên chọn món ăn nào nhỉ",bạn ơi tôi nên chọn món ăn nào nhỉ
2,ăn gì nhanh gọn vậy,ăn gì nhanh gọn vậy
3,gợi ý món dễ làm đi nha,gợi ý món dễ làm đi nha
4,"cho mình hỏi, Tôi cần một gợi ý về món ăn với",cho mình hỏi tôi cần một gợi ý về món ăn với


### Bước 2: Thay thế bằng mô hình BERT (Transformers)
Chúng ta sẽ cài đặt thư viện cần thiết và sử dụng `Sentence-Transformers` để dễ dàng tích hợp BERT vào pipeline phân loại.

In [5]:
!pip install -q sentence-transformers

### Bước 2: Huấn luyện Intent với BERT và Tiền xử lý
Chúng ta sẽ sử dụng `paraphrase-multilingual-MiniLM-L12-v2` để hỗ trợ tiếng Việt tốt nhất.

In [6]:
import re
import string
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Hàm tiền xử lý
def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 2. Chuẩn bị dữ liệu (bao gồm cả class 'other' để tránh lỗi 1 class)
X_raw = df_intent['text'].apply(preprocess_text).tolist()
y = df_intent['intent'].tolist()

extra_samples = ["xin chào", "hi", "tạm biệt", "kết thúc", "hướng dẫn giúp mình"]
extra_labels = ["other"] * len(extra_samples)

X_raw.extend(extra_samples)
y.extend(extra_labels)

# 3. Tải mô hình BERT
bert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 4. Chuyển đổi văn bản thành Embeddings
print("Đang trích xuất đặc trưng bằng BERT...")
X_embeddings = bert_model.encode(X_raw)

# 5. Chia dữ liệu
X_train, X_test, y_train, y_test = train_test_split(X_embeddings, y, test_size=0.2, random_state=42)

# 6. Huấn luyện Classifier
intent_classifier = LogisticRegression(max_iter=1000)
intent_classifier.fit(X_train, y_train)

print(f"Huấn luyện hoàn tất. Độ chính xác: {intent_classifier.score(X_test, y_test):.4f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Đang trích xuất đặc trưng bằng BERT...
Huấn luyện hoàn tất. Độ chính xác: 1.0000


In [7]:
def predict_intent_bert(text):
    cleaned = preprocess_text(text)
    vector = bert_model.encode([cleaned])
    return intent_classifier.predict(vector)[0]

# Test thử
query = "Cho mình hỏi có món gì ngon từ thịt gà không?"
print(f"Input: {query}")
print(f"Predicted Intent: {predict_intent_bert(query)}")

Input: Cho mình hỏi có món gì ngon từ thịt gà không?
Predicted Intent: recommend_dish


### Bước 3: Trích xuất thực thể (Entity Extraction) & Mapping VI-EN
Chúng ta sẽ sử dụng bộ dữ liệu `entity_dataset_400.json` để tìm các từ khóa nguyên liệu trong câu hỏi.

In [8]:
import re

# Bảng mapping thực thể mở rộng
entity_map = {
    'gà': {'en': 'chicken', 'type': 'ingredient'},
    'thịt gà': {'en': 'chicken', 'type': 'ingredient'},
    'bò': {'en': 'beef', 'type': 'ingredient'},
    'thịt bò': {'en': 'beef', 'type': 'ingredient'},
    'heo': {'en': 'pork', 'type': 'ingredient'},
    'tôm': {'en': 'shrimp', 'type': 'ingredient'},
    'rau xà lách': {'en': 'lettuce', 'type': 'ingredient'},
    'cà chua': {'en': 'tomato', 'type': 'ingredient'},
    'trứng': {'en': 'egg', 'type': 'ingredient'},
    'cá': {'en': 'fish', 'type': 'ingredient'},
    'hành': {'en': 'onion', 'type': 'ingredient'},
    'tỏi': {'en': 'garlic', 'type': 'ingredient'},
    'ớt': {'en': 'chilli', 'type': 'ingredient'},
    'rau cải': {'en': 'cabbage', 'type': 'ingredient'},
    'bắp cải': {'en': 'cabbage', 'type': 'ingredient'},
    'nấm': {'en': 'mushrooms', 'type': 'ingredient'},
    'phô mai': {'en': 'cheese', 'type': 'ingredient'},
    'xúc xích': {'en': 'sausage', 'type': 'ingredient'},
    'đậu': {'en': 'beans', 'type': 'ingredient'},
    'khoai tây': {'en': 'potato', 'type': 'ingredient'},
    'cà rốt': {'en': 'carrot', 'type': 'ingredient'},
    'dưa leo': {'en': 'cucumber', 'type': 'ingredient'},
    'rau': {'en': 'vegetable', 'type': 'ingredient'},
    'mì': {'en': 'noodles', 'type': 'ingredient'},
    'chanh': {'en': 'lemon', 'type': 'ingredient'},
    'bột mì': {'en': 'flour', 'type': 'ingredient'},
    'nghệ': {'en': 'turmeric', 'type': 'ingredient'},
    'sả': {'en': 'lemongrass', 'type': 'ingredient'},
    'lúa mì': {'en': 'wheat', 'type': 'ingredient'},
    'gạo': {'en': 'rice', 'type': 'ingredient'}
}

def extract_entities(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)

    found_ingredients = []
    found_dishes = []

    # Sắp xếp theo độ dài giảm dần để tránh khớp 'gà' trước 'thịt gà'
    sorted_keywords = sorted(entity_map.keys(), key=len, reverse=True)

    for kw in sorted_keywords:
        pattern = rf"\b{re.escape(kw)}\b"
        match = re.search(pattern, text)

        if match:
            info = entity_map[kw]
            entity_obj = {
                "vi": kw,
                "en": info["en"]
            }

            if info["type"] == "ingredient":
                found_ingredients.append(entity_obj)
            else:
                found_dishes.append(entity_obj)

            # Loại bỏ phần đã khớp để tránh trùng lặp
            text = re.sub(pattern, " ", text)

    return {
        "ingredients": found_ingredients,
        "dishes": found_dishes
    }

# Test thử
test_str = "Mình muốn nấu món thịt gà với rau và chanh"
result = extract_entities(test_str)
print(f"Input: {test_str}")
print(f"Kết quả: {result}")

Input: Mình muốn nấu món thịt gà với rau và chanh
Kết quả: {'ingredients': [{'vi': 'thịt gà', 'en': 'chicken'}, {'vi': 'chanh', 'en': 'lemon'}, {'vi': 'rau', 'en': 'vegetable'}], 'dishes': []}


### Bước 4: Gọi API TheMealDB và Xử lý kết quả
Sau khi có nguyên liệu tiếng Anh, chúng ta sẽ truy vấn API để lấy danh sách món ăn.

In [9]:
import requests

def get_recipes_from_api(ingredient_en):
    url = f"https://www.themealdb.com/api/json/v1/1/filter.php?i={ingredient_en}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data.get('meals', [])
    return []

# Demo gọi API với thực thể vừa tìm được
ingredient = 'tomato'
meals = get_recipes_from_api(ingredient)
print(f"Tìm thấy {len(meals)} món ăn liên quan đến '{ingredient}'")
if meals:
    for m in meals[:3]:
        print(f"- {m['strMeal']}")

Tìm thấy 32 món ăn liên quan đến 'tomato'
- Algerian Bouzgene Berber Bread with Roasted Pepper Sauce
- Arepa Pabellón
- Beef Empanadas


### Bước 4.1: Hệ thống Gợi ý Đa nguyên liệu & Xếp hạng (Multi-ingredient Ranking)
Logic này cho phép tìm kiếm món ăn dựa trên nhiều nguyên liệu cùng lúc và chấm điểm dựa trên độ phù hợp.

In [10]:
import requests
import re

# Cache để tránh gọi API trùng lặp
meal_cache = {}

def get_meal_details_cached(meal_id):
    if meal_id in meal_cache:
        return meal_cache[meal_id]
    url = f"https://www.themealdb.com/api/json/v1/1/lookup.php?i={meal_id}"
    try:
        res = requests.get(url)
        if res.status_code == 200:
            data = res.json()
            meals = data.get('meals')
            if meals:
                detail = meals[0]
                meal_cache[meal_id] = detail
                return detail
    except:
        pass
    return {}

def recommend_multi_ingredients(found_ingredients, criteria='ngon'):
    if not found_ingredients: return []

    user_ingredients_en = [ing['en'].lower() for ing in found_ingredients]
    all_meals_summary = {}

    # 1. Thu thập danh sách ID món ăn
    for ing in found_ingredients:
        meals = get_recipes_from_api(ing['en'])
        if not meals: continue
        for m in meals:
            m_id = m['idMeal']
            all_meals_summary[m_id] = all_meals_summary.get(m_id, 0) + 1

    if not all_meals_summary:
        return []

    # Lấy top 15 món tiềm năng
    top_ids = sorted(all_meals_summary.items(), key=lambda x: x[1], reverse=True)[:15]

    ranked_list = []
    for m_id, match_count in top_ids:
        details = get_meal_details_cached(m_id)
        if not details: continue

        # Trích xuất nguyên liệu thực tế từ API
        actual_ingredients = []
        for i in range(1, 21):
            ing_name = details.get(f'strIngredient{i}')
            if ing_name and ing_name.strip():
                actual_ingredients.append(ing_name.lower())

        # Đếm số lượng nguyên liệu khớp
        real_matches = 0
        for user_ing in user_ingredients_en:
            if any(user_ing in act_ing for act_ing in actual_ingredients):
                real_matches += 1

        # Tính điểm
        ing_score = (real_matches / len(user_ingredients_en)) * 100
        instructions = details.get('strInstructions', '')
        num_steps = len(instructions.split('.'))

        criteria_score = 0
        if criteria == 'nhanh':
            criteria_score = max(0, 50 - (num_steps * 2) - len(actual_ingredients))
        elif criteria == 'healthy':
            healthy_items = ['vegetable', 'chicken', 'fish', 'salad', 'spinach', 'broccoli', 'tomato']
            h_count = sum(1 for x in healthy_items if any(x in act for act in actual_ingredients))
            criteria_score = min(50, h_count * 10)
        else:
            criteria_score = 20 if 'sauce' in instructions.lower() else 10
            criteria_score += min(30, len(actual_ingredients) * 2)

        details['final_score'] = ing_score + criteria_score
        details['explanation'] = f"Khớp {real_matches}/{len(user_ingredients_en)} nguyên liệu."
        ranked_list.append(details)

    return sorted(ranked_list, key=lambda x: x['final_score'], reverse=True)

### Bước 4.2: Hàm lấy Công thức chi tiết (Recipe Details)
Hàm này sẽ lấy thông tin chi tiết của một món ăn cụ thể và định dạng lại nội dung (nguyên liệu, cách nấu).

In [11]:
def get_full_recipe(meal_id):
    # Đã sửa: Gọi đúng tên hàm get_meal_details_cached
    details = get_meal_details_cached(meal_id)
    if not details:
        return "Không tìm thấy thông tin chi tiết cho món này."

    name = details.get('strMeal')
    category = details.get('strCategory')
    area = details.get('strArea')
    instructions = details.get('strInstructions')

    # Lấy danh sách nguyên liệu và định lượng
    ingredients_list = []
    for i in range(1, 21):
        ingredient = details.get(f'strIngredient{i}')
        measure = details.get(f'strMeasure{i}')
        if ingredient and ingredient.strip():
            ingredients_list.append(f"- {ingredient} ({measure})")

    full_recipe = f"### Món ăn: {name}\n"
    full_recipe += f"**Loại:** {category} | **Xuất xứ:** {area}\n\n"
    full_recipe += "**Nguyên liệu:**\n" + "\n".join(ingredients_list) + "\n\n"
    full_recipe += "**Cách chế biến:**\n" + instructions

    return full_recipe

try:
    sample_meals = get_recipes_from_api('chicken')
    if sample_meals:
        top_meal_id = sample_meals[0]['idMeal']
        print(get_full_recipe(top_meal_id))
except Exception as e:
    print(f"Lỗi: {e}")

### Món ăn: Brown Stew Chicken
**Loại:** Chicken | **Xuất xứ:** Jamaican

**Nguyên liệu:**
- Chicken (1 whole)
- Tomato (1 chopped)
- Onions (2 chopped)
- Garlic Clove (2 chopped)
- Red Pepper (1 chopped)
- Carrots (1 chopped)
- Lime (1)
- Thyme (2 tsp)
- Allspice (1 tsp )
- Soy Sauce (2 tbs)
- Cornstarch (2 tsp)
- Coconut Milk (2 cups )
- Vegetable Oil (1 tbs)

**Cách chế biến:**
Squeeze lime over chicken and rub well. Drain off excess lime juice.
Combine tomato, scallion, onion, garlic, pepper, thyme, pimento and soy sauce in a large bowl with the chicken pieces. Cover and marinate at least one hour.
Heat oil in a dutch pot or large saucepan. Shake off the seasonings as you remove each piece of chicken from the marinade. Reserve the marinade for sauce.
Lightly brown the chicken a few pieces at a time in very hot oil. Place browned chicken pieces on a plate to rest while you brown the remaining pieces.
Drain off excess oil and return the chicken to the pan. Pour the marinade over the 

In [12]:
!pip install -q deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00


In [13]:
from deep_translator import GoogleTranslator

def translate_to_vi(text):
    try:
        if not text: return ""
        return GoogleTranslator(source='en', target='vi').translate(text)
    except Exception as e:
        return text

def get_full_recipe_vi(meal_id):
    # Đã sửa: Gọi đúng tên hàm get_meal_details_cached
    details = get_meal_details_cached(meal_id)
    if not details:
        return "Không tìm thấy thông tin chi tiết."

    name_en = details.get('strMeal')
    instructions_en = details.get('strInstructions')

    # Dịch tiêu đề và hướng dẫn
    name_vi = translate_to_vi(name_en)
    instructions_vi = translate_to_vi(instructions_en)

    # Lấy và dịch nguyên liệu
    ingredients_list = []
    for i in range(1, 21):
        ing = details.get(f'strIngredient{i}')
        measure = details.get(f'strMeasure{i}')
        if ing and ing.strip():
            combined = f"{measure} {ing}"
            ingredients_list.append(f"- {translate_to_vi(combined)}")

    full_recipe = f"### Món ăn: {name_vi} ({name_en})\n"
    full_recipe += f"**Loại:** {details.get('strCategory')} | **Xuất xứ:** {details.get('strArea')}\n\n"
    full_recipe += "**Nguyên liệu:**\n" + "\n".join(ingredients_list) + "\n\n"
    full_recipe += "**Cách chế biến:**\n" + instructions_vi

    return full_recipe

try:
    sample_meals = get_recipes_from_api('chicken')
    if sample_meals:
        top_id = sample_meals[0]['idMeal']
        print(get_full_recipe_vi(top_id))
except Exception as e:
    print(f"Lỗi dịch: {e}")

### Món ăn: Gà hầm nâu (Brown Stew Chicken)
**Loại:** Chicken | **Xuất xứ:** Jamaican

**Nguyên liệu:**
- 1 con gà nguyên con
- 1 quả cà chua xắt nhỏ
- 2 củ hành tây xắt nhỏ
- 2 tép tỏi băm nhỏ
- 1 quả ớt đỏ cắt nhỏ
- 1 củ cà rốt cắt nhỏ
- 1 quả chanh
- 2 muỗng cà phê húng tây
- 1 muỗng cà phê hạt tiêu
- 2 muỗng canh nước tương
- 2 muỗng cà phê bột bắp
- 2 cốc nước cốt dừa
- 1 muỗng canh dầu thực vật

**Cách chế biến:**
Vắt chanh lên gà và chà xát đều. Xả hết nước cốt chanh dư thừa.
Kết hợp cà chua, hành lá, hành tây, tỏi, hạt tiêu, húng tây, ớt và nước tương vào tô lớn cùng với các miếng thịt gà. Che và ướp ít nhất một giờ.
Đun nóng dầu trong nồi Hà Lan hoặc chảo lớn. Lắc bỏ gia vị khi lấy từng miếng thịt gà ra khỏi nước xốt. Dự trữ nước xốt cho nước sốt.
Chiên từng miếng gà một vài miếng trong dầu thật nóng. Đặt các miếng thịt gà đã chiên vàng lên đĩa để nghỉ trong khi chiên các miếng còn lại.
Xả bớt dầu thừa và cho gà trở lại chảo. Đổ nước xốt lên gà và thêm cà rốt. Khuấy và nấu trê

### Bước 5: Chatbot Pipeline Hoàn chỉnh (End-to-End)
Kết nối tất cả các thành phần theo trình tự từ nhận input đến trả kết quả tiếng Việt.

In [20]:
import re

last_recommendations = []
translation_cache = {}

def get_cached_translation(text):
    global translation_cache
    if not text: return ""
    if len(translation_cache) > 1000: translation_cache.clear()
    if text not in translation_cache:
        translation_cache[text] = translate_to_vi(text)
    return translation_cache[text]

def handle_selection(user_input_low):
    global last_recommendations
    if not last_recommendations: return None
    # Sửa lại regex với tiếng Việt chuẩn
    match_num = re.search(r'(?:chọn|lấy|món|số|cho mình|cho tôi|giúp)\s*(\d+)', user_input_low)
    idx = -1
    if not match_num and user_input_low.isdigit():
        idx = int(user_input_low) - 1
    elif match_num:
        idx = int(match_num.group(1)) - 1
    if 0 <= idx < len(last_recommendations):
        return get_full_recipe_vi(last_recommendations[idx]['idMeal'])
    return None

def handle_recommendation(entities, user_input_low):
    global last_recommendations
    criteria = 'ngon'
    if any(w in user_input_low for w in ['nhanh', 'gọn', 'lẹ']): criteria = 'nhanh'
    elif any(w in user_input_low for w in ['healthy', 'lành mạnh', 'giảm cân']): criteria = 'healthy'

    results = recommend_multi_ingredients(entities['ingredients'], criteria)
    if not results:
        last_recommendations = []
        return "Rất tiếc, mình chưa tìm thấy món nào khớp với các nguyên liệu này."

    last_recommendations = results[:5]
    res_text = f"GỢI Ý CHO BẠN ({criteria}):\n\n"
    for i, m in enumerate(last_recommendations):
        name_vi = get_cached_translation(m['strMeal'])
        res_text += f"{i+1}. **{name_vi}** ({m['strMeal']})\n   → _{m['explanation']}_\n"
    res_text += "\nBạn hãy gõ số thứ tự hoặc tên món để xem công thức nhé!"
    return res_text

### Bước 6: Thử nghiệm Chatbot Tương tác
Sử dụng ô nhập liệu dưới đây để trò chuyện với Chatbot.

In [15]:
# Sửa lỗi đồng bộ: Chạy ô này để cập nhật toàn bộ Logic Chatbot vào bộ nhớ

def predict_intent_bert(text):
    cleaned = preprocess_text(text)
    vector = bert_model.encode([cleaned])
    return intent_classifier.predict(vector)[0]

def get_full_recipe_vi(meal_id):
    details = get_meal_details_cached(meal_id)
    if not details: return "Không tìm thấy thông tin."
    name_vi = translate_to_vi(details.get('strMeal'))
    instr_vi = translate_to_vi(details.get('strInstructions'))
    ing_list = []
    for i in range(1, 21):
        ing = details.get(f'strIngredient{i}')
        mea = details.get(f'strMeasure{i}')
        if ing and ing.strip():
            ing_list.append(f"- {translate_to_vi(f'{mea} {ing}')}")
    return f"### {name_vi}\n**Nguyên liệu:**\n" + "\n".join(ing_list) + f"\n\n**Cách làm:**\n{instr_vi}"

def chatbot_response(user_input):
    user_input_low = user_input.lower().strip()
    selection = handle_selection(user_input_low)
    if selection: return selection

    entities = extract_entities(user_input)
    intent = predict_intent_bert(user_input)

    if entities['ingredients'] or intent == 'recommend_dish':
        if not entities['ingredients']:
            return "Bạn muốn nấu với nguyên liệu gì nhỉ?"
        return handle_recommendation(entities, user_input_low)
    return "Xin lỗi, mình chưa hiểu ý bạn. Bạn thử nhập nguyên liệu nhé (vd: gà, bò)."

print("✅ Toàn bộ hệ thống đã được đồng bộ hóa!")

✅ Toàn bộ hệ thống đã được đồng bộ hóa!


In [22]:
def chat_loop():
    print("🤖 Chatbot gợi ý món ăn (gõ 'exit' để thoát)\n")

    while True:
        user_msg = input("🧑 Bạn: ")

        if user_msg.lower() in ["exit", "quit"]:
            print("👋 Tạm biệt!")
            break

        if not user_msg.strip():
            print("⚠️ Bạn chưa nhập gì!\n")
            continue

        try:
            response = chatbot_response(user_msg)
            print("🤖 Chatbot:", response, "\n")

        except Exception as e:
            print("❌ Lỗi:", str(e), "\n")

# Gọi hàm
chat_loop()

🤖 Chatbot gợi ý món ăn (gõ 'exit' để thoát)

🧑 Bạn: có hành gà ớt nấm thì ăn gì ngon
🤖 Chatbot: GỢI Ý CHO BẠN (ngon):

1. **Lẩu gà nấm** (Chicken & mushroom Hotpot)
   → _Khớp 3/4 nguyên liệu._
2. **Gà Alfredo Primavera** (Chicken Alfredo Primavera)
   → _Khớp 3/4 nguyên liệu._
3. **Gà Handi** (Chicken Handi)
   → _Khớp 3/4 nguyên liệu._
4. **Mandi gà** (Chicken Mandi)
   → _Khớp 3/4 nguyên liệu._
5. **Bigos (Thợ săn hầm)** (Bigos (Hunters Stew))
   → _Khớp 2/4 nguyên liệu._

Bạn hãy gõ số thứ tự hoặc tên món để xem công thức nhé! 

🧑 Bạn: số 1
🤖 Chatbot: ### Lẩu gà nấm
**Nguyên liệu:**
- 50g bơ
- 1 củ hành tây xắt nhỏ
- 100g Nấm
- 40g bột mì
- 1 viên nước luộc gà
- nhúm nhục đậu khấu
- nhúm bột mù tạt
- 250g thịt gà
- 2 nắm ngô ngọt
- 2 củ khoai tây lớn
- 1 núm bơ

**Cách làm:**
Làm nóng lò ở nhiệt độ 200C/180C quạt/gas 6. Cho bơ vào nồi cỡ vừa và đặt trên lửa vừa. Thêm hành tây và để nấu trong 5 phút, thỉnh thoảng khuấy. Cho nấm vào nồi cùng hành tây.

Khi hành tây và nấm gần chín, c